# Neural Feature Explorer & Decoder Lab

**Data:** immutable, fold-independent DEV feature cache only; no raw
neural bundle and no CONFIRM access.

[Release provenance](https://github.com/c-lin-chunyi/nma-project-data-analysis/releases/tag/neural-dev-features-v1-29482249873)

Run the cells from top to bottom. This notebook is self-contained: it
does not clone a repository. Its first code cell installs only missing
dependencies. The complete feature cache is about 24.3 MiB compressed.


## 0. Prepare the Colab runtime


In [ ]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from importlib.util import find_spec

requirements = [
    ("pandas", "pandas", "pandas"),
    ("pyarrow", "pyarrow", "pyarrow"),
    ("h5py", "h5py", "h5py"),
    ("sklearn", "scikit-learn", "scikit-learn"),
    ("matplotlib", "matplotlib", "matplotlib"),
    ("seaborn", "seaborn", "seaborn"),
    ("ipywidgets", "ipywidgets", "ipywidgets"),
]
install = []
for module, package, distribution in requirements:
    if find_spec(module) is None:
        install.append(package)

if install:
    print("Installing missing Colab dependencies:", ", ".join(install))
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", *install
    ])
else:
    print("All notebook dependencies are already available.")

versions = {}
for _, _, distribution in requirements:
    try:
        versions[distribution] = version(distribution)
    except PackageNotFoundError:
        versions[distribution] = "installed in this cell; restart only if import fails"
print("Runtime versions:", versions)


## 1. Verified release loader

Embedded for one-file Colab use.


In [ ]:
"""Verified, cache-aware readers for the immutable public DEV releases.

The notebook helpers intentionally avoid AllenSDK and the 16 GB neural bundle.
They consume only the compact behavioral scan and analysis-ready feature cache.
"""

from __future__ import annotations

import hashlib
import json
import os
import shutil
import sys
import tarfile
import urllib.error
import urllib.request
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Mapping

import h5py
import numpy as np
import pandas as pd

REPOSITORY = "c-lin-chunyi/nma-project-data-analysis"
BEHAVIORAL_TAG = "behavioral-v3.1-29482141350"
FEATURE_TAG = "neural-dev-features-v1-29482249873"
BEHAVIORAL_ARCHIVE = "behavioral-v3.1-scan.tar.gz"
FEATURE_MANIFEST = "feature-cache-manifest.json"
FEATURE_NAMES = (
    "events_baselined_post",
    "events_unbaselined_pre",
    "events_unbaselined_post",
    "events_baselined_full_pre",
    "dff_baselined_post",
)

BEHAVIORAL_REQUIRED_COLUMNS = {
    "trial_labels": {
        "behavior_session_id",
        "mouse_id",
        "trial_id",
        "trial_index",
        "late_hit",
        "early_hit",
        "miss",
        "aborted",
        "engaged_A",
        "engaged_B",
        "engaged_A_hysteretic",
        "keep_A",
        "keep_B",
        "keep_A_hysteretic",
    },
    "session_scan": {
        "behavior_session_id",
        "mouse_id",
        "n_trials",
        "late_hit_B",
        "miss_B",
        "behavioral_eligible",
    },
}


class ReleaseDataError(RuntimeError):
    """Raised when a release asset is unavailable, unsafe, or invalid."""


def default_cache_dir() -> Path:
    override = os.environ.get("NMA_RELEASE_CACHE")
    if override:
        return Path(override).expanduser()
    if "google.colab" in sys.modules or Path("/content").is_dir():
        return Path("/content/nma-release-cache")
    return Path.home() / ".cache" / "nma-project-data-analysis"


def release_url(tag: str, asset: str) -> str:
    return f"https://github.com/{REPOSITORY}/releases/download/{tag}/{asset}"


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def parse_sha256sums(path: Path) -> dict[str, str]:
    result: dict[str, str] = {}
    for line in path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) >= 2 and len(parts[0]) == 64:
            result[parts[-1].lstrip("*")] = parts[0].lower()
    if not result:
        raise ReleaseDataError(f"No SHA-256 entries found in {path}")
    return result


def _progress(name: str, block: int, block_size: int, total: int) -> None:
    if total <= 0:
        return
    downloaded = min(block * block_size, total)
    pct = 100 * downloaded / total
    print(f"\r{name}: {downloaded / 2**20:.1f}/{total / 2**20:.1f} MiB ({pct:5.1f}%)",
          end="", flush=True)
    if downloaded >= total:
        print()


def _copy_or_download(
    tag: str,
    asset: str,
    destination: Path,
    *,
    expected_sha256: str | None = None,
    source_dir: Path | None = None,
    show_progress: bool = True,
) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file() and (
        expected_sha256 is None or sha256_file(destination) == expected_sha256.lower()
    ):
        return destination
    if destination.exists():
        destination.unlink()

    partial = destination.with_name(destination.name + ".part")
    if partial.exists():
        partial.unlink()
    try:
        if source_dir is not None:
            source = Path(source_dir) / asset
            if not source.is_file():
                raise ReleaseDataError(f"Missing local release asset: {source}")
            shutil.copyfile(source, partial)
        else:
            callback = (
                (lambda block, size, total: _progress(asset, block, size, total))
                if show_progress else None
            )
            urllib.request.urlretrieve(release_url(tag, asset), partial, callback)
        if expected_sha256 is not None:
            actual = sha256_file(partial)
            if actual != expected_sha256.lower():
                raise ReleaseDataError(
                    f"SHA-256 mismatch for {asset}: expected {expected_sha256}, got {actual}"
                )
        os.replace(partial, destination)
    except (OSError, urllib.error.URLError) as exc:
        if partial.exists():
            partial.unlink()
        if isinstance(exc, ReleaseDataError):
            raise
        raise ReleaseDataError(f"Could not obtain {asset}: {exc}") from exc
    except Exception:
        if partial.exists():
            partial.unlink()
        raise
    return destination


def _safe_extract_tar(archive: Path, destination: Path) -> None:
    """Extract regular files/directories without trusting archive paths or links."""
    destination.mkdir(parents=True, exist_ok=True)
    base = destination.resolve()
    with tarfile.open(archive, "r:*") as tar:
        for member in tar:
            relative = Path(member.name)
            if relative.is_absolute() or ".." in relative.parts:
                raise ReleaseDataError(f"Unsafe path in {archive.name}: {member.name}")
            target = (destination / relative).resolve()
            try:
                target.relative_to(base)
            except ValueError as exc:
                raise ReleaseDataError(
                    f"Archive member escapes destination: {member.name}"
                ) from exc
            if member.isdir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            if not member.isfile():
                raise ReleaseDataError(
                    f"Unsupported link/device in {archive.name}: {member.name}"
                )
            source = tar.extractfile(member)
            if source is None:
                raise ReleaseDataError(f"Could not read {member.name} from {archive.name}")
            target.parent.mkdir(parents=True, exist_ok=True)
            with source, target.open("wb") as output:
                shutil.copyfileobj(source, output)


def _extract_once(archive: Path, destination: Path, expected_sha256: str) -> None:
    marker = destination / ".archive-sha256"
    if marker.is_file() and marker.read_text().strip() == expected_sha256:
        return
    if destination.exists():
        shutil.rmtree(destination)
    _safe_extract_tar(archive, destination)
    marker.write_text(expected_sha256 + "\n")


@dataclass(frozen=True)
class BehaviorScan:
    tag: str
    root: Path
    tables: Mapping[str, pd.DataFrame]
    manifest: Mapping[str, Any]

    def __getitem__(self, name: str) -> pd.DataFrame:
        return self.tables[name]

    @property
    def trial_labels(self) -> pd.DataFrame:
        return self.tables["trial_labels"]

    @property
    def session_scan(self) -> pd.DataFrame:
        return self.tables["session_scan"]


def _behavior_table_name(path: Path) -> str:
    return path.stem.lstrip("_")


def _validate_behavior(scan: BehaviorScan, *, strict_public: bool) -> None:
    if scan.manifest.get("schema") != "behavioral-v3.1":
        raise ReleaseDataError("Unexpected behavioral manifest schema")
    for name, required in BEHAVIORAL_REQUIRED_COLUMNS.items():
        if name not in scan.tables:
            raise ReleaseDataError(f"Behavioral scan is missing {name}.parquet")
        missing = required - set(scan.tables[name].columns)
        if missing:
            raise ReleaseDataError(f"{name}.parquet is missing columns: {sorted(missing)}")
    if strict_public:
        if int(scan.manifest.get("n_dev_sessions", -1)) != 50:
            raise ReleaseDataError("Public behavioral release must contain 50 DEV sessions")
        if scan.session_scan["behavior_session_id"].nunique() != 50:
            raise ReleaseDataError("Behavioral session table does not contain 50 sessions")


def load_behavioral_scan(
    cache_dir: str | Path | None = None,
    *,
    source_dir: str | Path | None = None,
    show_progress: bool = True,
) -> BehaviorScan:
    """Download, verify, extract, and read the compact behavioral DEV scan."""
    cache = Path(cache_dir) if cache_dir is not None else default_cache_dir()
    source_value = source_dir or os.environ.get("NMA_BEHAVIORAL_SOURCE_DIR")
    local_source = Path(source_value) if source_value is not None else None
    root = cache / "behavioral" / BEHAVIORAL_TAG
    sums_path = _copy_or_download(
        BEHAVIORAL_TAG, "SHA256SUMS", root / "SHA256SUMS",
        source_dir=local_source, show_progress=show_progress,
    )
    sums = parse_sha256sums(sums_path)
    expected = sums.get(BEHAVIORAL_ARCHIVE)
    if expected is None:
        raise ReleaseDataError(f"SHA256SUMS does not cover {BEHAVIORAL_ARCHIVE}")
    archive = _copy_or_download(
        BEHAVIORAL_TAG, BEHAVIORAL_ARCHIVE, root / BEHAVIORAL_ARCHIVE,
        expected_sha256=expected, source_dir=local_source, show_progress=show_progress,
    )
    manifest_sha = sums.get("behavioral-manifest.json")
    manifest_path = _copy_or_download(
        BEHAVIORAL_TAG, "behavioral-manifest.json", root / "behavioral-manifest.json",
        expected_sha256=manifest_sha, source_dir=local_source, show_progress=show_progress,
    )
    extracted = root / "scan"
    _extract_once(archive, extracted, expected)
    try:
        tables = {
            _behavior_table_name(path): pd.read_parquet(path)
            for path in sorted(extracted.glob("*.parquet"))
        }
        manifest = json.loads(manifest_path.read_text())
    except Exception as exc:
        raise ReleaseDataError(f"Could not read behavioral scan files: {exc}") from exc
    scan = BehaviorScan(BEHAVIORAL_TAG, extracted, tables, manifest)
    _validate_behavior(scan, strict_public=local_source is None)
    return scan


@dataclass(frozen=True)
class FeatureMatrix:
    values: np.ndarray
    trial_ids: np.ndarray
    cell_ids: np.ndarray
    name: str
    experiment_id: int


class FeatureCache:
    """Lazy reader over the extracted, experiment-level feature cache."""

    feature_names = FEATURE_NAMES

    def __init__(
        self,
        root: Path,
        manifest: Mapping[str, Any],
        validation: Mapping[str, Any],
        experiments: pd.DataFrame,
        *,
        strict_public: bool = True,
    ) -> None:
        self.root = Path(root)
        self.manifest = dict(manifest)
        self.validation = dict(validation)
        self.source_experiments = experiments.copy()
        meta_rows = []
        self._meta: dict[int, dict[str, Any]] = {}
        for path in sorted(self.root.glob("*.feature-meta.json")):
            meta = json.loads(path.read_text())
            identity = dict(meta["identity"])
            oeid = int(identity["ophys_experiment_id"])
            self._meta[oeid] = meta
            meta_rows.append({
                **identity,
                "n_trials": int(meta["n_trials"]),
                "n_cells": int(meta["n_cells"]),
            })
        self.index = pd.DataFrame(meta_rows).sort_values(
            ["mouse_id", "ophys_container_id", "ophys_experiment_id"]
        ).reset_index(drop=True)
        self._validate(strict_public=strict_public)

    @property
    def experiment_ids(self) -> list[int]:
        return self.index["ophys_experiment_id"].astype(int).tolist()

    def meta(self, experiment_id: int) -> dict[str, Any]:
        try:
            return self._meta[int(experiment_id)]
        except KeyError as exc:
            raise KeyError(f"Unknown feature-cache experiment: {experiment_id}") from exc

    def labels(self, experiment_id: int) -> pd.DataFrame:
        path = self.root / f"{int(experiment_id)}.labels.parquet"
        return pd.read_parquet(path)

    def q2(self, experiment_id: int) -> pd.DataFrame:
        path = self.root / f"{int(experiment_id)}.q2.parquet"
        return pd.read_parquet(path)

    def matrix(self, experiment_id: int, name: str = FEATURE_NAMES[0]) -> FeatureMatrix:
        if name not in FEATURE_NAMES:
            raise KeyError(f"Unknown feature {name!r}; choose one of {FEATURE_NAMES}")
        oeid = int(experiment_id)
        with h5py.File(self.root / f"{oeid}.features.h5", "r") as h5:
            values = h5[name][:]
            trial_ids = h5["trial_id"][:]
            cell_ids = h5["cell_specimen_id"][:]
        return FeatureMatrix(values, trial_ids, cell_ids, name, oeid)

    def _validate(self, *, strict_public: bool) -> None:
        if self.manifest.get("schema") != "neural-dev-feature-cache-v1":
            raise ReleaseDataError("Unexpected feature-cache manifest schema")
        if not bool(self.validation.get("complete")):
            raise ReleaseDataError(
                f"Feature-cache validation is not complete: {self.validation.get('failures')}"
            )
        if self.index.empty:
            raise ReleaseDataError("No feature experiments were extracted")
        expected = int(self.manifest.get("n_active_experiments", -1))
        if len(self.index) != expected:
            raise ReleaseDataError(
                f"Expected {expected} active experiments, found {len(self.index)}"
            )
        if strict_public and (
            expected != 50 or int(self.manifest.get("n_containers", -1)) != 10
        ):
            raise ReleaseDataError("Public feature cache must contain 50 experiments/10 containers")
        for oeid in self.experiment_ids:
            files = [
                self.root / f"{oeid}.features.h5",
                self.root / f"{oeid}.labels.parquet",
                self.root / f"{oeid}.q2.parquet",
            ]
            if not all(path.is_file() for path in files):
                missing = [path.name for path in files if not path.is_file()]
                raise ReleaseDataError(f"Experiment {oeid} is missing files: {missing}")
            labels = self.labels(oeid)
            q2 = self.q2(oeid)
            with h5py.File(files[0], "r") as h5:
                missing_features = set(FEATURE_NAMES) - set(h5.keys())
                if missing_features:
                    raise ReleaseDataError(
                        f"Experiment {oeid} is missing HDF5 datasets: {sorted(missing_features)}"
                    )
                trial_ids = h5["trial_id"][:]
                n_cells = len(h5["cell_specimen_id"])
                for feature in FEATURE_NAMES:
                    if h5[feature].shape != (len(trial_ids), n_cells):
                        raise ReleaseDataError(
                            f"Experiment {oeid} has invalid shape for {feature}"
                        )
            if not np.array_equal(labels["trial_id"].to_numpy(), trial_ids):
                raise ReleaseDataError(f"Experiment {oeid} label/HDF5 trial IDs do not align")
            if not np.array_equal(q2["trial_id"].to_numpy(), trial_ids):
                raise ReleaseDataError(f"Experiment {oeid} Q2/HDF5 trial IDs do not align")


def load_feature_cache(
    cache_dir: str | Path | None = None,
    *,
    source_dir: str | Path | None = None,
    show_progress: bool = True,
) -> FeatureCache:
    """Download and verify all compact feature shards, then return a lazy reader."""
    cache = Path(cache_dir) if cache_dir is not None else default_cache_dir()
    source_value = source_dir or os.environ.get("NMA_FEATURE_SOURCE_DIR")
    local_source = Path(source_value) if source_value is not None else None
    root = cache / "features" / FEATURE_TAG
    sums_path = _copy_or_download(
        FEATURE_TAG, "SHA256SUMS", root / "SHA256SUMS",
        source_dir=local_source, show_progress=show_progress,
    )
    sums = parse_sha256sums(sums_path)
    metadata_names = [
        FEATURE_MANIFEST,
        "feature-cache-validation.json",
        "dev_experiments.csv",
    ]
    metadata: dict[str, Path] = {}
    for name in metadata_names:
        expected = sums.get(name)
        if expected is None:
            raise ReleaseDataError(f"SHA256SUMS does not cover {name}")
        metadata[name] = _copy_or_download(
            FEATURE_TAG, name, root / name, expected_sha256=expected,
            source_dir=local_source, show_progress=show_progress,
        )
    manifest = json.loads(metadata[FEATURE_MANIFEST].read_text())
    validation = json.loads(metadata["feature-cache-validation.json"].read_text())
    extracted = root / "cache"
    parts = manifest.get("parts", [])
    if not parts:
        raise ReleaseDataError("Feature manifest lists no shards")
    for part in parts:
        name = str(part["name"])
        expected = str(part["sha256"]).lower()
        archive = _copy_or_download(
            FEATURE_TAG, name, root / "archives" / name,
            expected_sha256=expected, source_dir=local_source, show_progress=show_progress,
        )
        marker = extracted / f".{name}.sha256"
        if not marker.is_file() or marker.read_text().strip() != expected:
            _safe_extract_tar(archive, extracted)
            marker.parent.mkdir(parents=True, exist_ok=True)
            marker.write_text(expected + "\n")
    experiments = pd.read_csv(metadata["dev_experiments.csv"])
    try:
        return FeatureCache(
            extracted,
            manifest,
            validation,
            experiments,
            strict_public=local_source is None,
        )
    except ReleaseDataError:
        raise
    except Exception as exc:
        raise ReleaseDataError(f"Could not read feature-cache files: {exc}") from exc


## 2. Typed single-experiment decoder

Embedded for one-file Colab use.


In [ ]:
"""A typed, single-experiment Q1 decoder for the educational Colab notebook."""

from __future__ import annotations

import hashlib
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler



@dataclass(frozen=True)
class DecoderConfig:
    k: int | None = 50
    C: float = 1e-4
    n_seeds: int = 10
    n_folds: int = 5
    purge: int = 10
    cv: str = "blocked"

    @property
    def exploratory(self) -> bool:
        return not (
            self.k == 50
            and self.C == 1e-4
            and self.n_seeds == 10
            and self.n_folds == 5
            and self.purge == 10
            and self.cv == "blocked"
        )

    def validate(self) -> None:
        if self.k is not None and self.k < 1:
            raise ValueError("k must be positive or None for all cells")
        if self.C <= 0:
            raise ValueError("C must be positive")
        if self.n_seeds < 1:
            raise ValueError("n_seeds must be positive")
        if self.n_folds < 2:
            raise ValueError("n_folds must be at least 2")
        if self.purge < 0:
            raise ValueError("purge must be non-negative")
        if self.cv not in {"blocked", "random"}:
            raise ValueError("cv must be 'blocked' or 'random'")


@dataclass
class DecoderResult:
    status: str
    reason: str | None
    config: DecoderConfig
    experiment_id: int
    feature_name: str
    seed_metrics: pd.DataFrame = field(default_factory=pd.DataFrame)
    fold_metrics: pd.DataFrame = field(default_factory=pd.DataFrame)
    oof: pd.DataFrame = field(default_factory=pd.DataFrame)
    cell_summary: pd.DataFrame = field(default_factory=pd.DataFrame)

    @property
    def mean_auc(self) -> float:
        if self.seed_metrics.empty:
            return float("nan")
        return float(self.seed_metrics["auc"].mean())


def contiguous_purged_folds(
    raw_index: np.ndarray, n_blocks: int = 5, purge: int = 10
) -> list[tuple[np.ndarray, np.ndarray]]:
    raw_index = np.asarray(raw_index, dtype=int)
    order = np.argsort(raw_index)
    result = []
    for test in np.array_split(order, n_blocks):
        if not len(test):
            continue
        low, high = int(raw_index[test].min()), int(raw_index[test].max())
        train = np.setdiff1d(order, test)
        adjacent = (
            ((raw_index[train] >= low - purge) & (raw_index[train] < low))
            | ((raw_index[train] > high) & (raw_index[train] <= high + purge))
        )
        result.append((train[~adjacent], test))
    return result


def deterministic_cell_indices(
    n_cells: int, k: int | None, seed: int, experiment_id: int
) -> np.ndarray | None:
    if k is None:
        return np.arange(n_cells)
    if n_cells < k:
        return None
    digest = hashlib.sha256(f"{int(experiment_id)}:{int(k)}:{int(seed)}".encode()).digest()
    rng = np.random.default_rng(int.from_bytes(digest[:8], "big"))
    return np.sort(rng.choice(n_cells, int(k), replace=False))


def _empty_result(
    reason: str,
    config: DecoderConfig,
    matrix: FeatureMatrix,
) -> DecoderResult:
    return DecoderResult(
        status="nonestimable",
        reason=reason,
        config=config,
        experiment_id=matrix.experiment_id,
        feature_name=matrix.name,
    )


def run_q1_decoder(
    matrix: FeatureMatrix,
    labels: pd.DataFrame,
    config: DecoderConfig | None = None,
) -> DecoderResult:
    """Fit the registered single-session late-hit-vs-miss decoder.

    This follows the v3.2/v3.3 Q1 session defaults but does not perform the
    authoritative mouse-level aggregation.
    """
    config = config or DecoderConfig()
    config.validate()
    required = {
        "trial_id", "trial_index", "engaged_B", "keep_B", "late_hit", "miss",
    }
    missing = required - set(labels.columns)
    if missing:
        return _empty_result(f"missing_label_columns:{','.join(sorted(missing))}", config, matrix)
    aligned = labels.set_index("trial_id").reindex(matrix.trial_ids).reset_index()
    if aligned["trial_index"].isna().any():
        return _empty_result("trial_alignment_failed", config, matrix)
    mask = (
        aligned["engaged_B"].fillna(False).astype(bool)
        & aligned["keep_B"].fillna(False).astype(bool)
        & (
            aligned["late_hit"].fillna(False).astype(bool)
            | aligned["miss"].fillna(False).astype(bool)
        )
    )
    X = np.asarray(matrix.values[mask.to_numpy()], dtype=float)
    selected_labels = aligned.loc[mask].reset_index(drop=True)
    y = selected_labels["late_hit"].astype(int).to_numpy()
    raw = selected_labels["trial_index"].astype(int).to_numpy()
    if len(np.unique(y)) < 2:
        return _empty_result("insufficient_classes", config, matrix)
    if not np.isfinite(X).all():
        return _empty_result("nonfinite_features", config, matrix)
    if config.k is not None and X.shape[1] < config.k:
        return _empty_result("low_cells", config, matrix)

    seed_rows: list[dict] = []
    fold_rows: list[dict] = []
    coefficient_rows: list[dict] = []
    oof_columns: dict[str, np.ndarray] = {}
    selected_by_seed: dict[int, np.ndarray] = {}

    for seed in range(config.n_seeds):
        cell_index = deterministic_cell_indices(
            X.shape[1], config.k, seed, matrix.experiment_id
        )
        if cell_index is None:
            return _empty_result("low_cells", config, matrix)
        selected_by_seed[seed] = cell_index
        X_seed = X[:, cell_index]
        scores = np.full(len(y), np.nan)
        if config.cv == "blocked":
            splits = contiguous_purged_folds(raw, config.n_folds, config.purge)
        else:
            class_counts = np.bincount(y, minlength=2)
            if int(class_counts.min()) < config.n_folds:
                return _empty_result("random_cv_class_support_nonestimable", config, matrix)
            splits = list(
                StratifiedKFold(
                    config.n_folds, shuffle=True, random_state=seed
                ).split(X_seed, y)
            )
        if len(splits) != config.n_folds:
            return _empty_result("fold_construction_failed", config, matrix)

        for fold, (train, test) in enumerate(splits):
            train_classes = np.bincount(y[train], minlength=2)
            test_classes = np.bincount(y[test], minlength=2)
            fold_row = {
                "seed": seed,
                "fold": fold,
                "cv": config.cv,
                "n_train": len(train),
                "n_test": len(test),
                "train_negative": int(train_classes[0]),
                "train_positive": int(train_classes[1]),
                "test_negative": int(test_classes[0]),
                "test_positive": int(test_classes[1]),
                "test_raw_min": int(raw[test].min()),
                "test_raw_max": int(raw[test].max()),
                "estimable": bool((train_classes > 0).all()),
            }
            fold_rows.append(fold_row)
            if not fold_row["estimable"]:
                return _empty_result("temporal_support_nonestimable", config, matrix)
            scaler = StandardScaler().fit(X_seed[train])
            model = LogisticRegression(
                C=config.C,
                penalty="l2",
                class_weight="balanced",
                solver="liblinear",
                random_state=seed,
                max_iter=2000,
            )
            model.fit(scaler.transform(X_seed[train]), y[train])
            scores[test] = model.decision_function(scaler.transform(X_seed[test]))
            for local_cell, coefficient in zip(cell_index, model.coef_[0]):
                coefficient_rows.append({
                    "seed": seed,
                    "fold": fold,
                    "cell_index": int(local_cell),
                    "cell_id": int(matrix.cell_ids[local_cell]),
                    "coefficient": float(coefficient),
                })
        if not np.isfinite(scores).all():
            return _empty_result("score_nonestimable", config, matrix)
        seed_rows.append({
            "seed": seed,
            "auc": float(roc_auc_score(y, scores)),
            "n_trials": len(y),
            "n_positive": int(y.sum()),
            "n_negative": int((1 - y).sum()),
            "n_cells": len(cell_index),
        })
        oof_columns[f"score_seed_{seed}"] = scores

    oof = selected_labels[["trial_id", "trial_index", "late_hit", "miss"]].copy()
    oof["y"] = y
    for name, values in oof_columns.items():
        oof[name] = values
    score_columns = [name for name in oof.columns if name.startswith("score_seed_")]
    oof["mean_score"] = oof[score_columns].mean(axis=1)

    coefficients = pd.DataFrame(coefficient_rows)
    cell_summary_rows = []
    for index, cell_id in enumerate(matrix.cell_ids):
        selected_seeds = sum(index in indices for indices in selected_by_seed.values())
        values = coefficients.loc[
            coefficients["cell_index"].eq(index), "coefficient"
        ]
        cell_summary_rows.append({
            "cell_index": index,
            "cell_id": int(cell_id),
            "selection_frequency": selected_seeds / config.n_seeds,
            "median_coefficient": float(values.median()) if len(values) else np.nan,
            "mean_abs_coefficient": float(values.abs().mean()) if len(values) else np.nan,
        })
    return DecoderResult(
        status="estimable",
        reason=None,
        config=config,
        experiment_id=matrix.experiment_id,
        feature_name=matrix.name,
        seed_metrics=pd.DataFrame(seed_rows),
        fold_metrics=pd.DataFrame(fold_rows),
        oof=oof,
        cell_summary=pd.DataFrame(cell_summary_rows),
    )


## 3. Load and verify all ten feature shards


In [ ]:
from contextlib import nullcontext
from functools import lru_cache

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import ipywidgets as widgets
from IPython.display import HTML, display
from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve
from sklearn.preprocessing import StandardScaler

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass

sns.set_theme(style="whitegrid", context="notebook")


def display_matplotlib_figure(figure):
    # Emit a native PNG payload that is reliable inside Colab Output widgets.
    try:
        figure.tight_layout()
    except Exception:
        pass
    display(figure)
    plt.close(figure)


cache = load_feature_cache()
index = cache.index.copy()
display(HTML(
    "<div style='padding:12px;border-left:5px solid #d97706;background:#fff7ed'>"
    "<b>DEV-only educational explorer.</b> It uses fold-independent features from "
    f"<code>{FEATURE_TAG}</code>, never downloads raw neural bundles, and never "
    "requests CONFIRM data.</div>"
))
print(
    f"Loaded {len(index):,} active experiments across "
    f"{index.ophys_container_id.nunique():,} containers and "
    f"{index.mouse_id.nunique():,} mice."
)


@lru_cache(maxsize=24)
def experiment_data(experiment_id, feature):
    experiment_id = int(experiment_id)
    matrix = cache.matrix(experiment_id, feature)
    labels = cache.labels(experiment_id).set_index("trial_id").reindex(
        matrix.trial_ids
    ).reset_index()
    q2 = cache.q2(experiment_id).set_index("trial_id").reindex(
        matrix.trial_ids
    ).reset_index()
    return matrix, labels, q2


def outcome_labels(labels):
    return np.select(
        [labels.late_hit, labels.early_hit, labels.miss, labels.aborted],
        ["late hit", "early hit", "miss", "abort"],
        default="other",
    )


## 4. Cohort browser


In [ ]:
def metric_cards(items):
    cards = "".join(
        f"<div style='flex:1;min-width:150px;padding:14px;border:1px solid #e5e7eb;"
        f"border-radius:10px;background:white'><div style='font-size:12px;color:#6b7280'>"
        f"{label}</div><div style='font-size:26px;font-weight:700'>{value}</div></div>"
        for label, value in items
    )
    display(HTML(f"<div style='display:flex;gap:10px;flex-wrap:wrap'>{cards}</div>"))

metric_cards([
    ("Active experiments", f"{len(index):,}"),
    ("DEV mice", f"{index.mouse_id.nunique():,}"),
    ("Aligned trials", f"{index.n_trials.sum():,}"),
    ("Cells per experiment", f"{index.n_cells.min()}–{index.n_cells.max()}"),
])

figure, axis = plt.subplots(figsize=(11, 6))
sns.scatterplot(
    data=index,
    x="n_trials",
    y="n_cells",
    hue="project_code",
    style="session_type",
    size="n_cells",
    sizes=(70, 360),
    alpha=0.8,
    ax=axis,
)
axis.set(
    title="Feature-cache coverage",
    xlabel="Aligned trials",
    ylabel="Cells",
)
axis.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
display_matplotlib_figure(figure)

hierarchy = index.sort_values(
    ["mouse_id", "ophys_container_id", "ophys_experiment_id"]
).copy()
hierarchy["row_label"] = (
    hierarchy.mouse_id.astype(str)
    + " / "
    + hierarchy.ophys_container_id.astype(str)
)
hierarchy["experiment_order"] = (
    hierarchy.groupby(["mouse_id", "ophys_container_id"]).cumcount() + 1
)
figure, axis = plt.subplots(figsize=(12, 7))
sns.scatterplot(
    data=hierarchy,
    x="experiment_order",
    y="row_label",
    hue="session_type",
    style="session_type",
    size="n_trials",
    sizes=(80, 360),
    alpha=0.85,
    ax=axis,
)
axis.set(
    title="Mouse → container → experiment hierarchy (size = trials)",
    xlabel="Experiment order within container",
    ylabel="Mouse / container",
)
axis.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
display_matplotlib_figure(figure)

coverage = pd.crosstab(index.mouse_id.astype(str), index.session_type)
figure, axis = plt.subplots(figsize=(12, 5))
sns.heatmap(
    coverage,
    annot=True,
    fmt="d",
    cmap="crest",
    linewidths=0.5,
    cbar_kws={"label": "Experiments"},
    ax=axis,
)
axis.set(
    title="Session-type coverage by mouse",
    xlabel="Session type",
    ylabel="Mouse",
)
display_matplotlib_figure(figure)


## 5. Interactive feature explorer


In [ ]:
FEATURE_LABELS = {
    "events_baselined_post": "Events · baseline-subtracted · [0, 0.30)s",
    "events_unbaselined_pre": "Events · unbaselined · [-1, 0)s",
    "events_unbaselined_post": "Events · unbaselined · [0, 0.30)s",
    "events_baselined_full_pre": "Events · baseline-subtracted · full pre",
    "dff_baselined_post": "dF/F · baseline-subtracted · [0, 0.30)s",
}

mouse_dd = widgets.Dropdown(
    options=sorted(index.mouse_id.astype(int).unique()),
    description="Mouse:", style={"description_width": "initial"},
)
container_dd = widgets.Dropdown(
    description="Container:", style={"description_width": "initial"}
)
experiment_dd = widgets.Dropdown(
    description="Experiment:", style={"description_width": "initial"}
)
feature_dd = widgets.Dropdown(
    options=[(label, name) for name, label in FEATURE_LABELS.items()],
    value="events_baselined_post",
    description="Representation:", style={"description_width": "initial"},
    layout={"width": "430px"},
)
scale_dd = widgets.ToggleButtons(
    options=[("Cell z-score", "z"), ("Raw", "raw")],
    value="z", description="Heatmap:",
)
cell_sort_dd = widgets.Dropdown(
    options=[("Late-hit effect", "effect"), ("Variance", "variance")],
    value="effect", description="Cell order:", style={"description_width": "initial"},
)
matrix_out = widgets.Output()
geometry_out = widgets.Output()
decoder_out = widgets.Output()


def visible_widget_errors(output, function):
    # Keep callback failures visible instead of leaving a blank Colab panel.
    def wrapped(change=None):
        try:
            return function(change)
        except Exception as exc:
            output.outputs = ()
            with output:
                display(HTML(
                    "<b>Rendering failed.</b> The exception below is shown so "
                    "the notebook does not fail silently."
                ))
                print(f"{type(exc).__name__}: {exc}")
    return wrapped


def sync_containers(*_):
    rows = index[index.mouse_id.astype(int).eq(int(mouse_dd.value))]
    options = [int(value) for value in sorted(rows.ophys_container_id.astype(int).unique())]
    previous = container_dd.value
    container_dd.options = options
    container_dd.value = previous if previous in options else (options[0] if options else None)


def sync_experiments(*_):
    if container_dd.value is None:
        experiment_dd.options = []
        return
    rows = index[index.ophys_container_id.astype(int).eq(int(container_dd.value))]
    options = [
        (f"{int(row.ophys_experiment_id)} · {row.session_type}", int(row.ophys_experiment_id))
        for row in rows.sort_values("ophys_experiment_id").itertuples()
    ]
    previous = experiment_dd.value
    experiment_dd.options = options
    values = [value for _, value in options]
    experiment_dd.value = previous if previous in values else (values[0] if values else None)
    update_k_options()


def selected_effect(values, labels):
    late = labels.late_hit.fillna(False).to_numpy(bool)
    miss = labels.miss.fillna(False).to_numpy(bool)
    engaged = labels.engaged_B.fillna(False).to_numpy(bool)
    keep = labels.keep_B.fillna(False).to_numpy(bool)
    left = values[late & engaged & keep]
    right = values[miss & engaged & keep]
    if not len(left) or not len(right):
        return np.zeros(values.shape[1])
    pooled = np.sqrt((np.nanvar(left, axis=0) + np.nanvar(right, axis=0)) / 2)
    return np.divide(
        np.nanmean(left, axis=0) - np.nanmean(right, axis=0),
        pooled,
        out=np.zeros(values.shape[1]),
        where=np.isfinite(pooled) & (pooled > 0),
    )


def bootstrap_population(frame, groups, n_boot=500):
    rng = np.random.default_rng(20260717)
    rows = []
    for label in ["late hit", "miss", "early hit", "abort", "other"]:
        values = frame[np.asarray(groups) == label]
        if not len(values):
            continue
        means = np.array([
            rng.choice(values, len(values), replace=True).mean() for _ in range(n_boot)
        ])
        rows.append({
            "outcome": label,
            "mean": float(np.mean(values)),
            "low": float(np.quantile(means, 0.025)),
            "high": float(np.quantile(means, 0.975)),
            "n": len(values),
        })
    return pd.DataFrame(rows)


def render_matrix(*_, direct=False):
    if experiment_dd.value is None:
        return
    oeid = int(experiment_dd.value)
    matrix, labels, _ = experiment_data(oeid, feature_dd.value)
    values = matrix.values.astype(float)
    effect = selected_effect(values, labels)
    order = (
        np.argsort(np.abs(effect))[::-1]
        if cell_sort_dd.value == "effect"
        else np.argsort(np.nanvar(values, axis=0))[::-1]
    )
    order = order[: min(200, len(order))]
    trial_outcome = outcome_labels(labels)
    trial_order = np.lexsort((labels.trial_index.to_numpy(), trial_outcome))
    shown = values[trial_order][:, order]
    if scale_dd.value == "z":
        mean = np.nanmean(shown, axis=0)
        sd = np.nanstd(shown, axis=0)
        shown = np.divide(shown - mean, sd, out=np.zeros_like(shown), where=sd > 0)
    population = np.nanmean(values, axis=1)
    summary = bootstrap_population(population, trial_outcome)
    if not direct:
        matrix_out.outputs = ()
    with (nullcontext() if direct else matrix_out):
        display(HTML(
            f"<h3>Experiment {oeid}</h3><p>{FEATURE_LABELS[feature_dd.value]} · "
            f"{len(values):,} trials × {values.shape[1]:,} cells. "
            f"Heatmap shows the top {len(order)} cells.</p>"
        ))
        heat_frame = pd.DataFrame(
            shown,
            index=labels.trial_index.to_numpy()[trial_order],
            columns=[str(int(matrix.cell_ids[i])) for i in order],
        )
        figure, axis = plt.subplots(figsize=(15, 8))
        sns.heatmap(
            heat_frame,
            cmap="vlag",
            center=0,
            xticklabels=max(1, len(order) // 20),
            yticklabels=max(1, len(heat_frame) // 25),
            cbar_kws={
                "label": "Cell z-score" if scale_dd.value == "z" else "Response"
            },
            ax=axis,
        )
        axis.set(
            title="Trial × cell response matrix",
            xlabel="Cell specimen ID",
            ylabel="Raw trial index (grouped by outcome)",
        )
        display_matplotlib_figure(figure)

        figure, axis = plt.subplots(figsize=(9, 5))
        lower = summary["mean"] - summary.low
        upper = summary.high - summary["mean"]
        axis.bar(
            summary.outcome,
            summary["mean"],
            yerr=np.vstack([lower, upper]),
            color=sns.color_palette("deep", len(summary)),
            capsize=4,
        )
        for position, row in enumerate(summary.itertuples()):
            axis.annotate(
                f"n={row.n}",
                (position, row.high),
                xytext=(0, 6),
                textcoords="offset points",
                ha="center",
            )
        axis.set(
            title="Population response by outcome (bootstrap 95% CI)",
            xlabel="Outcome",
            ylabel="Mean across cells",
        )
        display_matplotlib_figure(figure)

        effect_table = pd.DataFrame({
            "cell_id": matrix.cell_ids,
            "effect": effect,
            "variance": np.nanvar(values, axis=0),
        }).sort_values("effect")
        effect_table["rank"] = np.arange(len(effect_table))
        figure, axis = plt.subplots(figsize=(11, 5))
        points = axis.scatter(
            effect_table["rank"],
            effect_table["effect"],
            c=effect_table["variance"],
            cmap="viridis",
            s=28,
            alpha=0.8,
        )
        figure.colorbar(points, ax=axis, label="Variance")
        axis.axhline(0, color="0.45", linewidth=1)
        axis.set(
            title="Per-cell standardized late-hit − miss effect",
            xlabel="Effect rank",
            ylabel="Standardized effect",
        )
        display_matplotlib_figure(figure)

        distribution = pd.DataFrame({
            "population_response": population,
            "outcome": trial_outcome,
        })
        figure, axis = plt.subplots(figsize=(9, 5))
        sns.violinplot(
            data=distribution,
            x="outcome",
            y="population_response",
            hue="outcome",
            inner="box",
            cut=0,
            legend=False,
            ax=axis,
        )
        axis.set(
            title="Trial-level population-response distributions",
            xlabel="Outcome",
            ylabel="Population response",
        )
        display_matplotlib_figure(figure)

        comparisons = [
            ("events_baselined_post", "dff_baselined_post"),
            ("events_unbaselined_pre", "events_unbaselined_post"),
        ]
        titles = ["Events post vs dF/F post", "Events pre vs events post"]
        figure, axes = plt.subplots(1, 2, figsize=(13, 5))
        for axis, title, (left_name, right_name) in zip(axes, titles, comparisons):
            left = experiment_data(oeid, left_name)[0].values.mean(axis=1)
            right = experiment_data(oeid, right_name)[0].values.mean(axis=1)
            correlation = np.corrcoef(left, right)[0, 1]
            sns.scatterplot(x=left, y=right, s=25, alpha=0.55, ax=axis)
            axis.set(
                title=f"{title} · r={correlation:.2f}",
                xlabel="Left population response",
                ylabel="Right population response",
            )
        figure.suptitle("Representation comparisons")
        display_matplotlib_figure(figure)


geometry_color_dd = widgets.Dropdown(
    options=["outcome", "engaged_B", "session_position"],
    value="outcome", description="PCA color:", style={"description_width": "initial"},
)


def render_geometry(*_, direct=False):
    if experiment_dd.value is None:
        return
    oeid = int(experiment_dd.value)
    matrix, labels, q2 = experiment_data(oeid, feature_dd.value)
    values = matrix.values.astype(float)
    finite = np.isfinite(values).all(axis=1)
    if finite.sum() < 3:
        if not direct:
            geometry_out.outputs = ()
        with (nullcontext() if direct else geometry_out):
            print("PCA nonestimable: fewer than three finite trials.")
        return
    scaled = StandardScaler().fit_transform(values[finite])
    n_components = min(10, scaled.shape[0], scaled.shape[1])
    pca = PCA(n_components=n_components).fit(scaled)
    scores = pca.transform(scaled)
    plot = pd.DataFrame({"PC1": scores[:, 0], "PC2": scores[:, 1]})
    plot["outcome"] = outcome_labels(labels)[finite]
    plot["engaged_B"] = labels.engaged_B.fillna(False).to_numpy()[finite].astype(str)
    plot["session_position"] = q2.session_position.to_numpy()[finite]
    population = np.nanmean(values, axis=1)
    joint = q2.copy()
    joint["population_response"] = population
    joint["outcome"] = outcome_labels(labels)
    if not direct:
        geometry_out.outputs = ()
    with (nullcontext() if direct else geometry_out):
        figure, axis = plt.subplots(figsize=(9, 6))
        color_column = geometry_color_dd.value
        sns.scatterplot(
            data=plot,
            x="PC1",
            y="PC2",
            hue=color_column,
            palette="viridis" if color_column == "session_position" else "deep",
            s=42,
            alpha=0.75,
            ax=axis,
        )
        axis.set_title(f"Trial geometry · {FEATURE_LABELS[feature_dd.value]}")
        axis.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
        display_matplotlib_figure(figure)

        figure, axis = plt.subplots(figsize=(9, 5))
        sns.barplot(
            x=np.arange(1, n_components + 1),
            y=pca.explained_variance_ratio_,
            color=sns.color_palette()[0],
            ax=axis,
        )
        axis.set(
            title="PCA scree plot",
            xlabel="Principal component",
            ylabel="Explained variance ratio",
        )
        display_matplotlib_figure(figure)

        covariates = [
            ("pre_change_pupil", "Pre-change pupil"),
            ("pre_change_running", "Pre-change running"),
            ("session_position", "Session position"),
        ]
        figure, axes = plt.subplots(1, 3, figsize=(15, 5))
        for axis, (name, label_text) in zip(axes, covariates):
            good = joint[name].notna() & joint.population_response.notna()
            sns.regplot(
                data=joint.loc[good],
                x=name,
                y="population_response",
                scatter_kws={"s": 18, "alpha": 0.45},
                line_kws={"color": "#c44e52"},
                ax=axis,
            )
            axis.set_title(label_text)
        figure.suptitle("Behavior–neural relationships (no imputation)")
        display_matplotlib_figure(figure)

        missing = (
            q2.isna().mean().sort_values(ascending=False).rename("missing_fraction")
            .reset_index().rename(columns={"index": "covariate"})
        )
        figure_height = max(5, 0.3 * len(missing))
        figure, axis = plt.subplots(figsize=(10, figure_height))
        sns.barplot(
            data=missing,
            x="missing_fraction",
            y="covariate",
            color=sns.color_palette()[0],
            ax=axis,
        )
        axis.set(
            title="Q2 covariate missingness",
            xlabel="Missing fraction",
            ylabel="Covariate",
        )
        display_matplotlib_figure(figure)


decoder_feature_dd = widgets.Dropdown(
    options=[(label, name) for name, label in FEATURE_LABELS.items()],
    value="events_baselined_post",
    description="Representation:", style={"description_width": "initial"},
    layout={"width": "430px"},
)
k_dd = widgets.Dropdown(description="K:", style={"description_width": "initial"})
c_dd = widgets.Dropdown(
    options=[("1e-5", 1e-5), ("1e-4 · registered", 1e-4), ("1e-3", 1e-3),
             ("1e-2", 1e-2), ("1e-1", 1e-1), ("1", 1.0), ("10", 10.0)],
    value=1e-4, description="C:", style={"description_width": "initial"},
)
seed_dd = widgets.Dropdown(
    options=[1, 5, 10], value=10, description="Seeds:",
    style={"description_width": "initial"},
)
cv_dd = widgets.Dropdown(
    options=[("Blocked + purge · registered", "blocked"),
             ("Random stratified · diagnostic", "random")],
    value="blocked", description="CV:", style={"description_width": "initial"},
)
run_button = widgets.Button(
    description="Run decoder", button_style="primary", icon="play"
)
decoder_cache = {}


def update_k_options(*_):
    if experiment_dd.value is None:
        return
    n_cells = int(
        index.loc[
            index.ophys_experiment_id.astype(int).eq(int(experiment_dd.value)), "n_cells"
        ].iloc[0]
    )
    options = [(str(k), k) for k in (20, 50, 100) if k <= n_cells]
    options.append((f"All ({n_cells}) · exploratory", None))
    k_dd.options = options
    if 50 <= n_cells:
        k_dd.value = 50


def decoder_key():
    return (
        int(experiment_dd.value), decoder_feature_dd.value, k_dd.value,
        float(c_dd.value), int(seed_dd.value), cv_dd.value,
    )


def render_decoder(_=None):
    key = decoder_key()
    oeid, feature, k, C, n_seeds, cv = key
    config = DecoderConfig(k=k, C=C, n_seeds=n_seeds, cv=cv)
    decoder_out.outputs = ()
    with decoder_out:
        label = "REGISTERED DEFAULTS" if not config.exploratory else "EXPLORATORY"
        color = "#166534" if not config.exploratory else "#b45309"
        display(HTML(
            f"<div style='padding:8px;border-left:4px solid {color}'><b>{label}</b> · "
            "Single-experiment educational analysis; not an authoritative mouse-level result."
            "</div>"
        ))
        print("Fitting train-only scalers and logistic models…")
    if key not in decoder_cache:
        matrix, labels, _ = experiment_data(oeid, feature)
        decoder_cache[key] = run_q1_decoder(matrix, labels, config)
    result = decoder_cache[key]
    decoder_out.outputs = ()
    with decoder_out:
        display(HTML(
            f"<div style='padding:8px;border-left:4px solid {color}'><b>{label}</b> · "
            "Single-experiment educational analysis; not an authoritative mouse-level result."
            "</div>"
        ))
        if result.status != "estimable":
            display(HTML(
                f"<h3>Decoder nonestimable</h3><code>{result.reason}</code>"
            ))
            return
        display(HTML(
            f"<h3>Mean seed AUC: {result.mean_auc:.3f}</h3>"
            f"<p>{len(result.oof):,} registered eligible trials · "
            f"{len(result.seed_metrics):,} deterministic seeds.</p>"
        ))
        figure, axis = plt.subplots(figsize=(9, 5))
        sns.barplot(
            data=result.seed_metrics,
            x="seed",
            y="auc",
            color=sns.color_palette()[0],
            ax=axis,
        )
        axis.axhline(0.5, linestyle="--", color="0.4")
        axis.set(
            title="Seed-level OOF AUC",
            xlabel="Seed",
            ylabel="OOF AUC",
            ylim=(0, 1),
        )
        display_matplotlib_figure(figure)

        fpr, tpr, _ = roc_curve(result.oof.y, result.oof.mean_score)
        figure, axis = plt.subplots(figsize=(7, 6))
        axis.plot(fpr, tpr, linewidth=2, label="Mean OOF score")
        axis.plot([0, 1], [0, 1], linestyle="--", color="0.5", label="Chance")
        axis.set(
            title="OOF ROC across mean seed scores",
            xlabel="False-positive rate",
            ylabel="True-positive rate",
            xlim=(0, 1),
            ylim=(0, 1),
        )
        axis.legend(frameon=False)
        display_matplotlib_figure(figure)

        fold = result.fold_metrics[
            result.fold_metrics.seed.eq(0)
        ].sort_values("fold")
        figure, axis = plt.subplots(figsize=(9, 5))
        axis.bar(
            fold.fold,
            fold.test_negative,
            label="miss",
            color=sns.color_palette("deep")[0],
        )
        axis.bar(
            fold.fold,
            fold.test_positive,
            bottom=fold.test_negative,
            label="late hit",
            color=sns.color_palette("deep")[1],
        )
        axis.set(
            title="Seed 0 temporal test-fold support",
            xlabel="Fold",
            ylabel="Trials",
        )
        axis.legend(frameon=False)
        display_matplotlib_figure(figure)

        temporal = result.oof.sort_values("trial_index").copy()
        temporal["outcome"] = temporal.y.map({0: "miss", 1: "late hit"})
        figure, axis = plt.subplots(figsize=(12, 5))
        sns.scatterplot(
            data=temporal,
            x="trial_index",
            y="mean_score",
            hue="outcome",
            style="outcome",
            s=32,
            alpha=0.75,
            ax=axis,
        )
        axis.axhline(0.5, linestyle="--", color="0.5")
        axis.set(
            title="OOF decision score over raw trial order",
            xlabel="Raw trial index",
            ylabel="Mean OOF score",
        )
        axis.legend(frameon=False)
        display_matplotlib_figure(figure)

        cells = result.cell_summary[
            result.cell_summary.selection_frequency.gt(0)
        ].sort_values(
            ["selection_frequency", "mean_abs_coefficient"], ascending=False
        ).head(60)
        figure, axis = plt.subplots(figsize=(10, 6))
        max_abs = max(float(cells.median_coefficient.abs().max()), 1e-12)
        sns.scatterplot(
            data=cells,
            x="selection_frequency",
            y="median_coefficient",
            size="mean_abs_coefficient",
            hue="median_coefficient",
            palette="vlag",
            hue_norm=(-max_abs, max_abs),
            sizes=(40, 320),
            alpha=0.8,
            ax=axis,
        )
        for row in cells.head(10).itertuples():
            axis.annotate(
                str(int(row.cell_id)),
                (row.selection_frequency, row.median_coefficient),
                xytext=(4, 4),
                textcoords="offset points",
                fontsize=8,
            )
        axis.axhline(0, color="0.45", linewidth=1)
        axis.set(
            title="Cell selection frequency and standardized coefficients",
            xlabel="Selection frequency",
            ylabel="Median standardized coefficient",
        )
        axis.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
        display_matplotlib_figure(figure)


# Establish a complete initial selection before registering observers. This
# avoids a burst of overlapping renders while Colab is mounting the widget tree.
sync_containers()
sync_experiments()
update_k_options()

matrix_controls = widgets.VBox([
    widgets.HBox([feature_dd, scale_dd, cell_sort_dd]),
    matrix_out,
])
geometry_controls = widgets.VBox([
    geometry_color_dd,
    geometry_out,
])
decoder_controls = widgets.VBox([
    widgets.HBox([decoder_feature_dd, k_dd, c_dd]),
    widgets.HBox([seed_dd, cv_dd, run_button]),
    decoder_out,
])
tabs = widgets.Tab(children=[matrix_controls, geometry_controls, decoder_controls])
for i, title in enumerate(["Matrix Explorer", "Trial Geometry", "Decoder Lab"]):
    tabs.set_title(i, title)

mouse_dd.observe(sync_containers, names="value")
container_dd.observe(sync_experiments, names="value")
experiment_dd.observe(update_k_options, names="value")
matrix_handler = visible_widget_errors(matrix_out, render_matrix)
geometry_handler = visible_widget_errors(geometry_out, render_geometry)
decoder_handler = visible_widget_errors(decoder_out, render_decoder)
feature_dd.observe(matrix_handler, names="value")
feature_dd.observe(geometry_handler, names="value")
scale_dd.observe(matrix_handler, names="value")
cell_sort_dd.observe(matrix_handler, names="value")
geometry_color_dd.observe(geometry_handler, names="value")
experiment_dd.observe(matrix_handler, names="value")
experiment_dd.observe(geometry_handler, names="value")
run_button.on_click(decoder_handler)

display(widgets.VBox([
    widgets.HTML("<h2>Select an experiment</h2>"),
    widgets.HBox([mouse_dd, container_dd, experiment_dd]),
    tabs,
]))

# The first render is emitted directly by the cell as native image/png. This
# avoids relying on nested widget-output capture during Colab's initial mount.
# Later selector changes render into the Output panels above.
render_matrix(direct=True)
render_geometry(direct=True)
